# 01 — Data Exploration & EDA

**MIT-BIH Arrhythmia Database** — 48 half-hour two-lead ECG recordings at 360 Hz.  
We explore the raw signals, beat annotations, and class distribution before any modelling.

**5 Classes:**
| Symbol | Beat Type | Label |
|--------|-----------|-------|
| N | Normal beat | 0 |
| L | Left Bundle Branch Block | 1 |
| R | Right Bundle Branch Block | 2 |
| A | Atrial Premature Contraction | 3 |
| V | Premature Ventricular Contraction | 4 |

In [ ]:
# Install dependencies (Colab)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'PyWavelets', 'scipy', 'matplotlib', 'seaborn', 'tqdm'], check=True)
print('Dependencies ready')

In [ ]:
import os, sys
import numpy as np
import wfdb
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.preprocess import BEAT_CLASSES, CLASS_NAMES, MITDB_RECORDS, FS

sns.set_theme(style='whitegrid', palette='muted')
print(f'Classes: {CLASS_NAMES}')

## 1. Download MIT-BIH Database

In [ ]:
DATA_DIR = '../data/mitdb'
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(DATA_DIR, '100.dat')):
    print('Downloading MIT-BIH ... (~100 MB, takes ~2-5 min)')
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)
    print('Download complete!')
else:
    print('MIT-BIH already downloaded.')

## 2. Inspect a Single Record

In [ ]:
record_id = '100'
record = wfdb.rdrecord(os.path.join(DATA_DIR, record_id))
ann    = wfdb.rdann(os.path.join(DATA_DIR, record_id), 'atr')

print(f'Record: {record_id}')
print(f'  Signals   : {record.sig_name}')
print(f'  Fs        : {record.fs} Hz')
print(f'  Duration  : {record.sig_len / record.fs / 60:.1f} min')
print(f'  Leads     : {record.n_sig}')
print(f'  Annotations: {len(ann.sample)} beats')

In [ ]:
# Plot 10 seconds of raw ECG with beat annotations
signal = record.p_signal
t = np.arange(signal.shape[0]) / FS

# find beats within first 10 s
mask = ann.sample < 10 * FS
beat_samples = ann.sample[mask]
beat_symbols = ann.symbol[mask]

colors_map = {'N': 'green', 'L': 'blue', 'R': 'orange', 'A': 'red', 'V': 'purple'}

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ch, ax in enumerate(axes):
    ax.plot(t[:10*FS], signal[:10*FS, ch], color='black', lw=0.7, label=record.sig_name[ch])
    for s, sym in zip(beat_samples, beat_symbols):
        if sym in colors_map:
            ax.axvline(s/FS, color=colors_map[sym], alpha=0.6, lw=1.2)
            ax.text(s/FS, signal[s, ch] + 0.1, sym, fontsize=7,
                    color=colors_map[sym], ha='center')
    ax.set_ylabel(record.sig_name[ch], fontsize=11)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)', fontsize=11)
axes[0].set_title(f'Record {record_id} — First 10 Seconds', fontsize=13, fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], color=c, lw=2, label=s)
                   for s, c in colors_map.items()]
axes[0].legend(handles=legend_elements, loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('../results/sample_ecg_record100.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Class Distribution Across All Records

In [ ]:
from tqdm.auto import tqdm

all_symbols = []
for rid in tqdm(MITDB_RECORDS, desc='Scanning records'):
    try:
        a = wfdb.rdann(os.path.join(DATA_DIR, rid), 'atr')
        all_symbols.extend(a.symbol)
    except:
        pass

counts = Counter(all_symbols)
target_counts = {s: counts[s] for s in BEAT_CLASSES if s in counts}

print('Beat counts for 5-class problem:')
total = sum(target_counts.values())
for sym, cnt in target_counts.items():
    print(f'  {sym} ({CLASS_NAMES[BEAT_CLASSES[sym]]:>20s}): {cnt:6d} ({cnt/total*100:.1f}%)')
print(f'  Total: {total}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [CLASS_NAMES[BEAT_CLASSES[s]] for s in target_counts]
values = list(target_counts.values())
colors = ['#2196F3', '#4CAF50', '#F44336', '#FF9800', '#9C27B0']

axes[0].bar(labels, values, color=colors, edgecolor='white')
axes[0].set_title('Beat Count by Class', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

axes[1].pie(values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=140, pctdistance=0.8)
axes[1].set_title('Class Distribution (Imbalance!)', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Severe class imbalance — N dominates. We address this with class-weighted loss.')

## 4. Beat Segment Examples (One per Class)

In [ ]:
from src.preprocess import bandpass_filter, remove_baseline, normalize_segment

# find one example of each class across records
examples = {}
HALF = 180

for rid in MITDB_RECORDS:
    if len(examples) == len(BEAT_CLASSES):
        break
    try:
        rec = wfdb.rdrecord(os.path.join(DATA_DIR, rid))
        ann = wfdb.rdann(os.path.join(DATA_DIR, rid), 'atr')
        sig = bandpass_filter(rec.p_signal[:, :2])
        sig = remove_baseline(sig)
        for sample, sym in zip(ann.sample, ann.symbol):
            if sym in BEAT_CLASSES and sym not in examples:
                if sample - HALF >= 0 and sample + HALF <= len(sig):
                    seg = sig[sample-HALF:sample+HALF, :].T  # (2, 360)
                    examples[sym] = normalize_segment(seg)
    except:
        pass

t = np.arange(360) / FS * 1000  # ms

fig, axes = plt.subplots(5, 2, figsize=(14, 14), sharey=False)
lead_names = ['MLII', 'V5']

for row, (sym, seg) in enumerate(sorted(examples.items(), key=lambda x: BEAT_CLASSES[x[0]])):
    label = CLASS_NAMES[BEAT_CLASSES[sym]]
    for col, (ax, lead) in enumerate(zip(axes[row], lead_names)):
        ax.plot(t, seg[col], color=colors[row], lw=1.5)
        ax.axvline(500, color='gray', lw=0.8, alpha=0.5, linestyle='--')
        ax.set_ylabel(f'{label}\n{lead}', fontsize=9)
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(f'Lead: {lead}', fontweight='bold')
        if row == 4:
            ax.set_xlabel('Time (ms)')

plt.suptitle('One Beat per Class — Preprocessed Segments (1 second @ 360 Hz)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/beat_examples_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Signal Statistics per Class

In [ ]:
import pandas as pd

# Load preprocessed arrays if available
X_path, y_path = '../data/X.npy', '../data/y.npy'
if os.path.exists(X_path):
    X = np.load(X_path)
    y = np.load(y_path)
    print(f'Loaded X={X.shape}, y={y.shape}')

    rows = []
    for cls_id, cls_name in enumerate(CLASS_NAMES):
        mask = y == cls_id
        segs = X[mask]  # (N, 2, 360)
        rows.append({
            'Class': cls_name,
            'Count': int(mask.sum()),
            'Mean Amplitude': f'{segs.mean():.4f}',
            'Std Amplitude': f'{segs.std():.4f}',
            'Max Amplitude': f'{segs.max():.4f}',
            'Min Amplitude': f'{segs.min():.4f}',
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
else:
    print('Run src/preprocess.py first to generate X.npy and y.npy')